In [2]:
from pathlib import Path
import re

root = Path().resolve().parent
src = root / 'application' / 'language' / 'english' / 'english_lang.php'
if not src.exists():
    raise FileNotFoundError(f'Legacy language file not found: {src}')

backup = src.with_suffix('.bak.php')
if not backup.exists():
    backup.write_bytes(src.read_bytes())

text = src.read_text(encoding='utf-8')
pattern = re.compile(r"^\s*\$lang\[['\"](?P<key>.*?)['\"]\]\s*=\s*['\"](?P<value>.*?)['\"]\s*;\s*$")
entries = []
for line in text.splitlines():
    m = pattern.match(line)
    if m:
        entries.append((m.group('key'), m.group('value')))

if not entries:
    raise ValueError('No language entries parsed from source file.')

out_dir = root / 'ci4-scaffold' / 'Language'
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / 'english_lang.php'
legacy_out = src

for target in [out_file, legacy_out]:
    with target.open('w', encoding='utf-8', newline='\n') as f:
        f.write('<?php\n')
        f.write('// Upgraded English language strings from legacy CI3 application/language/english/english_lang.php\n')
        f.write('return [\n')
        for key, value in entries:
            safe_value = value.replace('\\', '\\\\').replace("'", "\\'")
            f.write(f"    '{key}' => '{safe_value}',\n")
        f.write('];\n')

print('created:', out_file)
print('updated legacy file:', legacy_out)
print('backup saved:', backup)


created: C:\wamp64\www\multischool\ci4-scaffold\Language\english_lang.php
updated legacy file: C:\wamp64\www\multischool\application\language\english\english_lang.php
backup saved: C:\wamp64\www\multischool\application\language\english\english_lang.bak.php


In [ ]:
from pathlib import Path
import re

root = Path().resolve().parent
src = root / 'application' / 'config' / 'config.php'
text = src.read_text(encoding='utf-8')

config = {}
constants = {}

pattern = re.compile(r"^\s*\$config\[['\"](?P<key>[^'\"]+)['\"]\]\s*=\s*(?P<value>.+?);\s*$")
for line in text.splitlines():
    match = pattern.match(line)
    if not match:
        continue
    key = match.group('key')
    raw = match.group('value').strip()
    if raw.lower() == 'false':
        value = False
    elif raw.lower() == 'true':
        value = True
    elif raw.isdigit():
        value = int(raw)
    elif re.match(r"^0[0-7]+$", raw):
        value = int(raw, 8)
    elif raw.startswith('array(') and raw.endswith(')'):
        if raw == 'array()':
            value = []
        else:
            inner = raw[6:-1].strip()
            if not inner:
                value = []
            else:
                items = []
                for item in inner.split(','):
                    item = item.strip()
                    if item.lower() == 'false':
                        items.append(False)
                    elif item.lower() == 'true':
                        items.append(True)
                    elif item.startswith("'") and item.endswith("'"):
                        items.append(item[1:-1])
                    elif item.startswith('"') and item.endswith('"'):
                        items.append(item[1:-1])
                    elif item.isdigit():
                        items.append(int(item))
                    else:
                        items.append(item)
                value = items
    elif (raw.startswith("'") and raw.endswith("'")) or (raw.startswith('"') and raw.endswith('"')):
        value = raw[1:-1]
    else:
        value = raw
    config[key] = value

pattern_def = re.compile(r"^\s*define\(\s*['\"](?P<key>[^'\"]+)['\"]\s*,\s*(?P<value>.+?)\s*\);\s*$")
for line in text.splitlines():
    match = pattern_def.match(line)
    if not match:
        continue
    key = match.group('key')
    raw = match.group('value').strip()
    if raw.lower() == 'false':
        value = False
    elif raw.lower() == 'true':
        value = True
    elif raw.isdigit():
        value = int(raw)
    elif re.match(r"^0[0-7]+$", raw):
        value = int(raw, 8)
    elif (raw.startswith("'") and raw.endswith("'")) or (raw.startswith('"') and raw.endswith('"')):
        value = raw[1:-1]
    else:
        value = raw
    constants[key] = value

from pprint import pformat
export_config = pformat(config, indent=4, width=120)
export_constants = pformat(constants, indent=4, width=120)

out_config = f"<?php\nnamespace Config;\n\nuse CodeIgniter\\Config\\BaseConfig;\n\nclass ConfigStatic extends BaseConfig\n{{\n    /**\n     * Converted settings from application/config/config.php\n     * @var array\n     */\n    public array $settings = {export_config};\n\n    /**\n     * Legacy constants defined in application/config/config.php\n     * @var array\n     */\n    public array $constants = {export_constants};\n}}\n"

out_shim = "<?php\nnamespace Config;\n\nclass Config extends ConfigStatic\n{\n    // Compatibility shim for config('Config') usage.\n}\n"

out_dir = root / 'ci4-scaffold' / 'Config'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'ConfigStatic.php').write_text(out_config, encoding='utf-8')
(out_dir / 'Config.php').write_text(out_shim, encoding='utf-8')

print('wrote', out_dir / 'ConfigStatic.php')
print('wrote', out_dir / 'Config.php')
